# T11: Chaos Engineering

## What You'll Learn

Data pipelines break in predictable ways — nulls appear where they shouldn't, schemas drift overnight, and foreign keys stop matching. Spindle's chaos engine lets you inject these failures on purpose. In this notebook you will:

1. **Understand** the chaos testing philosophy
2. **Configure** a chaos engine at calm intensity
3. **Apply** schema drift (added/removed/renamed columns)
4. **Apply** value corruption (nulls, type mismatches, out-of-range values)
5. **Apply** referential chaos (broken foreign keys)
6. **Ramp up** to hurricane intensity
7. **Inspect** the corrupted data

## Prerequisites

- Python 3.9 or later
- `pip install sqllocks-spindle`

## Time Estimate

**~5 minutes** from start to finish.

In [1]:
# Uncomment the line below if you're running in an environment
# where sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Chaos Testing Philosophy

### Why inject chaos?

Most data pipelines are tested with clean, well-formed data. But production data is messy:

- A source system deploys a schema change and suddenly a column is renamed or removed
- A bug in an upstream system starts writing nulls into a NOT NULL column
- A data migration corrupts foreign-key relationships
- A decimal field starts receiving string values

If you only test with clean data, these failures will surprise you in production. Chaos engineering means **deliberately injecting failures** so you can verify your pipeline handles them gracefully — with clear error messages, proper quarantining, and no silent data loss.

### Spindle's Chaos Intensity Scale

| Intensity | Description |
|-----------|-------------|
| `calm` | Minimal chaos — a few nulls, one renamed column |
| `moderate` | Noticeable issues — multiple null injections, some type mismatches |
| `stormy` | Significant breakage — schema drift, broken FKs, widespread corruption |
| `hurricane` | Maximum chaos — everything breaks at once |

## Generate Clean Base Data

### What we're about to do
We'll generate a clean retail dataset that will serve as our baseline. We'll then apply chaos to copies of this data so we can compare before and after.

### Why this matters
You need a known-good baseline to measure the impact of chaos. By comparing the clean data to the corrupted version, you can see exactly what changed.

### What to expect
A clean retail dataset with zero nulls in required columns and valid foreign keys.

In [2]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())
print(f"\nBaseline null counts:")
for name, df in result.tables.items():
    null_count = df.isnull().sum().sum()
    print(f"  {name}: {null_count} nulls across {len(df.columns)} columns")

Spindle Generation Result
Schema: retail_3nf
Domain: retail
Mode:   3nf
Seed:   42
Time:   0.4s

Table                             Rows  Columns
---------------------------------------------
customer                         1,000        8
address                          1,500       10
product_category                    50        4
product                            500        6
promotion                          200        6
store                              150        5
order                            5,000        8
order_line                      12,500        8
return                             850        5
---------------------------------------------
TOTAL                           21,750

Baseline null counts:
  customer: 53 nulls across 8 columns
  address: 0 nulls across 10 columns
  product_category: 8 nulls across 4 columns
  product: 0 nulls across 6 columns
  promotion: 0 nulls across 6 columns
  store: 0 nulls across 5 columns
  order: 4570 nulls across 8 columns
  or

## Configure Chaos at Moderate Intensity

### What we're about to do
We'll create a `ChaosConfig` with `moderate` intensity. This will inject a noticeable but manageable amount of corruption — enough to test your pipeline's error handling without making the data completely unusable.

### Why this matters
Starting at moderate intensity lets you see the chaos patterns clearly. You can then ramp up to `stormy` or `hurricane` once your pipeline handles the basics.

### What to expect
A configured chaos engine ready to corrupt data.

In [3]:
from sqllocks_spindle.chaos import ChaosConfig, ChaosEngine

chaos_config = ChaosConfig(enabled=True, intensity="moderate", seed=99)
engine = ChaosEngine(chaos_config)

print(f"Chaos enabled: {chaos_config.enabled}")
print(f"Intensity: {chaos_config.intensity}")
print(f"Seed: {chaos_config.seed}")

Chaos enabled: True
Intensity: moderate
Seed: 99


## Apply Value Corruption

### What we're about to do
We'll apply value corruption to the `customer` table. This injects nulls into columns that should be NOT NULL, swaps data types, and introduces out-of-range values.

### Why this matters
Value corruption is the most common form of data quality issue in production. A field that was always populated suddenly has nulls. A numeric field receives a string. Testing for these issues before they hit production is essential.

### What to expect
The corrupted DataFrame will have more nulls than the original, and some values may have unexpected types or ranges.

In [4]:
# Work on a copy to preserve the original
customer_copy = result.tables["customer"].copy()

corrupted = engine.corrupt_dataframe(customer_copy, day=5)

original_nulls = result.tables["customer"].isnull().sum().sum()
corrupted_nulls = corrupted.isnull().sum().sum()

print(f"=== Value Corruption Results ===")
print(f"Original null count: {original_nulls}")
print(f"Corrupted null count: {corrupted_nulls}")
print(f"New nulls injected: {corrupted_nulls - original_nulls}")

# Show which columns gained nulls
print(f"\n=== Null Changes by Column ===")
for col in corrupted.columns:
    orig = result.tables["customer"][col].isnull().sum()
    corr = corrupted[col].isnull().sum()
    if corr > orig:
        print(f"  {col}: {orig} -> {corr} (+{corr - orig} nulls)")

=== Value Corruption Results ===
Original null count: 53
Corrupted null count: 53
New nulls injected: 0

=== Null Changes by Column ===


## Apply Schema Drift

### What we're about to do
We'll apply schema drift to the orders table. Schema drift includes:
- **Added columns**: new columns appear that your pipeline doesn't expect
- **Removed columns**: expected columns disappear
- **Renamed columns**: a column name changes (e.g., `order_date` becomes `orderDate`)

### Why this matters
Schema drift is inevitable in any system with multiple teams or external data sources. Your pipeline needs to detect drift, log it, and either adapt or fail gracefully — never silently drop data.

### What to expect
The drifted DataFrame will have a different set of columns than the original.

In [5]:
orders_copy = result.tables["order"].copy()
original_columns = set(orders_copy.columns)

drifted = engine.drift_schema(orders_copy, day=3)
drifted_columns = set(drifted.columns)

print(f"=== Schema Drift Results ===")
print(f"Original columns ({len(original_columns)}): {sorted(original_columns)}")
print(f"Drifted columns ({len(drifted_columns)}): {sorted(drifted_columns)}")

added = drifted_columns - original_columns
removed = original_columns - drifted_columns

if added:
    print(f"\nAdded columns: {sorted(added)}")
if removed:
    print(f"Removed columns: {sorted(removed)}")
if not added and not removed:
    print("\nNo column additions or removals (columns may have been renamed).")
    # Check for potential renames by comparing
    print(f"Check for renamed columns by comparing the lists above.")

=== Schema Drift Results ===
Original columns (8): ['customer_id', 'order_date', 'order_id', 'order_total', 'promotion_id', 'shipping_address_id', 'status', 'store_id']
Drifted columns (9): ['_chaos_extra_1046', 'customer_id', 'order_date', 'order_id', 'order_total', 'promotion_id', 'shipping_address_id', 'status', 'store_id']

Added columns: ['_chaos_extra_1046']


## Apply Referential Chaos

### What we're about to do
We'll break foreign-key relationships by corrupting key columns. For example, an order might reference a `customer_id` that doesn't exist in the customers table.

### Why this matters
Broken foreign keys cause JOINs to silently drop rows. If your pipeline doesn't check referential integrity, you'll end up with incomplete aggregations and misleading dashboards.

### What to expect
Some foreign-key values in the orders table will no longer match valid customer IDs.

In [6]:
valid_customer_ids = set(result.tables["customer"]["customer_id"])

# Check baseline integrity
baseline_orphans = result.tables["order"][~result.tables["order"]["customer_id"].isin(valid_customer_ids)]
print(f"Baseline orphaned orders: {len(baseline_orphans)}")

# Apply referential chaos — inject_referential_chaos takes a tables dict and day
tables_copy = {name: df.copy() for name, df in result.tables.items()}
broken_tables = engine.inject_referential_chaos(tables_copy, day=5)

# Check after chaos
broken_orders = broken_tables["order"]
chaos_orphans = broken_orders[~broken_orders["customer_id"].isin(valid_customer_ids)]
print(f"Orphaned orders after chaos: {len(chaos_orphans)}")
print(f"Orphan rate: {len(chaos_orphans) / len(broken_orders) * 100:.1f}%")

if len(chaos_orphans) > 0:
    print(f"\nExample orphaned customer_ids:")
    print(f"  {chaos_orphans['customer_id'].head().tolist()}")

Baseline orphaned orders: 0
Orphaned orders after chaos: 0
Orphan rate: 0.0%


## Ramp Up to Hurricane Intensity

### What we're about to do
We'll create a new chaos engine at `hurricane` intensity and apply it to the customer table. At this level, everything breaks: massive null injection, widespread type corruption, severe schema drift.

### Why this matters
Hurricane intensity is your worst-case scenario test. If your pipeline can survive this, it can handle anything production throws at it. This is the level you'd use for chaos testing before a major release.

### What to expect
Dramatically more corruption than the moderate run — significantly more nulls and potentially altered column types.

In [7]:
hurricane_config = ChaosConfig(enabled=True, intensity="hurricane", seed=99)
hurricane_engine = ChaosEngine(hurricane_config)

customer_hurricane = result.tables["customer"].copy()
wrecked = hurricane_engine.corrupt_dataframe(customer_hurricane, day=5)

original_nulls = result.tables["customer"].isnull().sum().sum()
hurricane_nulls = wrecked.isnull().sum().sum()

print(f"=== Hurricane Chaos Results ===")
print(f"Original null count: {original_nulls}")
print(f"Hurricane null count: {hurricane_nulls}")
print(f"New nulls injected: {hurricane_nulls - original_nulls}")
print(f"Corruption rate: {(hurricane_nulls - original_nulls) / (len(wrecked) * len(wrecked.columns)) * 100:.1f}%")

# Compare with moderate
print(f"\n=== Intensity Comparison ===")
print(f"Moderate: {corrupted_nulls - original_nulls} new nulls")
print(f"Hurricane: {hurricane_nulls - original_nulls} new nulls")

=== Hurricane Chaos Results ===
Original null count: 53
Hurricane null count: 53
New nulls injected: 0
Corruption rate: 0.0%

=== Intensity Comparison ===
Moderate: 0 new nulls
Hurricane: 0 new nulls


## Inspect Corrupted Data Side by Side

### What we're about to do
We'll display the first few rows of the original, moderate-chaos, and hurricane-chaos customer tables side by side so you can see the progression of corruption.

### Why this matters
Visual inspection helps you understand exactly what chaos does to your data. This makes it easier to write targeted validation rules and error handling.

### What to expect
Three DataFrames showing increasing levels of corruption.

In [8]:
print("=== Original (first 3 rows) ===")
print(result.tables["customer"].head(3).to_string())

print("\n=== Moderate Chaos (first 3 rows) ===")
print(corrupted.head(3).to_string())

print("\n=== Hurricane Chaos (first 3 rows) ===")
print(wrecked.head(3).to_string())

=== Original (first 3 rows) ===
   customer_id first_name  last_name                  email gender loyalty_tier                   signup_date is_active
0            1     Dillon  Macdonald     hyoung@example.org      M     Platinum 2022-12-24 11:42:37.230004110      true
1            2     Andrew       Paul  brianna69@example.com      M       Silver 2024-12-24 16:27:07.283292863      true
2            3    Jasmine   Schaefer     lsmith@example.net      F        Basic 2022-05-04 08:12:38.104288572      true

=== Moderate Chaos (first 3 rows) ===
  customer_id first_name  last_name                  email gender loyalty_tier                   signup_date is_active
0           1     Dillon  Macdonald     hyoung@example.org      M     Platinum 2022-12-24 11:42:37.230004110      true
1           2     Andrew       Paul  brianna69@example.com      M       Silver 2024-12-24 16:27:07.283292863      true
2           3    Jasmine   Schaefer     lsmith@example.net      F        Basic 2022-05-04 08

## What's Next?

You've injected chaos at multiple intensity levels — value corruption, schema drift, and referential breakage. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T12: Validation Gates** | Catch the chaos you just injected with automated gate checks |
| **T10: File-Drop Simulation** | Simulate daily file drops with manifests and late arrivals |
| **T09: Streaming Events** | Stream real-time events with configurable throughput |

Happy breaking things!